# 🏦 End-to-End Anti-Money Laundering (AML) Detection Pipeline
### *Written like a student. Explained like a senior engineer.*

---

## 1. Introduction

### 🟢 Simple Explanation
**Money laundering** is when criminals try to make "dirty" money (from crime) look like it came from a legal source. Imagine a drug dealer who opens a restaurant just to mix illegal cash with normal restaurant income — that's money laundering.

Banks are legally required to detect and report this. Our job is to build a machine learning system that automatically flags suspicious transactions.

---

### 🔵 Expert Insight: Why This Problem Is Hard

In a real AML system at a major bank, there are **two critical failure modes** — and both are expensive:

| Failure | What happens | Real-world consequence |
|---|---|---|
| **False Positive** (FP) | We flag a clean transaction as suspicious | Analyst wastes time investigating. If too many, analysts miss the 45-day regulatory deadline → **bank faces fines** |
| **False Negative** (FN) | We miss a real money laundering transaction | Criminal activity goes undetected → **bank faces massive regulatory fines, reputational damage, possible license revocation** |

This is why **accuracy is a terrible metric** here. If only 0.1% of transactions are illicit, a model that predicts "clean" for everything gets 99.9% accuracy — and catches zero criminals.

We care about **Recall** (catching real fraud) and **Precision** (not wasting analysts' time). The **F1-score** balances both.

---

### Our Pipeline
```
Raw Data → Preprocessing → PCA → K-Means Clustering → XGBoost → Continuous Learning
```

---
## 2. Import Libraries

### 🟢 Simple Explanation
Think of libraries as toolboxes. Instead of building everything from scratch, we borrow ready-made tools.

### 🔵 Expert Insight
We are deliberately keeping imports minimal. In production, you'd add monitoring libraries (MLflow, Evidently), feature stores, and data versioning tools. Here we focus on the core ML logic first — complexity is added when justified, not by default.

In [ ]:
# --- Standard data tools ---
import numpy as np                  # Math operations on arrays
import pandas as pd                 # Working with tables (DataFrames)

# --- Visualization ---
import matplotlib.pyplot as plt     # Basic plotting
import seaborn as sns               # Prettier plots

# --- Preprocessing ---
from sklearn.preprocessing import LabelEncoder, StandardScaler
# LabelEncoder: converts text categories to numbers
# StandardScaler: makes all numbers live on the same scale

# --- Dimensionality Reduction ---
from sklearn.decomposition import PCA
# PCA: squishes many features into fewer, keeping the most important information

# --- Clustering ---
from sklearn.cluster import KMeans
# KMeans: groups similar transactions together automatically

# --- XGBoost Classifier ---
import xgboost as xgb
# XGBoost: a powerful tree-based model great for tabular/financial data

# --- Model evaluation ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# --- Utility ---
import warnings
warnings.filterwarnings('ignore')   # Suppress non-critical warnings for cleaner output

# Make plots look nice
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print("✅ All libraries loaded successfully!")

---
## 3. Load Dataset

### 🟢 Simple Explanation
We use the **PaySim** dataset — a synthetic (fake-but-realistic) dataset that simulates mobile money transactions. It was built by researchers using real transaction patterns, then injected with simulated fraud.

You can download it from Kaggle: https://www.kaggle.com/datasets/ealaxi/paysim1

Download the file and place it in the same folder as this notebook, named `paysim.csv`.

### 🔵 Expert Insight
PaySim was designed to mirror real-world mobile money behavior in Africa. The fraud injection mechanism mimics two common laundering techniques: **type TRANSFER** (move money out) followed by **CASH_OUT** (withdraw). This mirrors real smurfing and layering patterns. The dataset's 6.3M rows also reflects the volume challenge in real AML systems — you can't manually review everything.

In [ ]:
# --- Load the PaySim dataset ---
# Make sure 'paysim.csv' is in the same folder as this notebook
# Download from: https://www.kaggle.com/datasets/ealaxi/paysim1

try:
    # Load a sample of 200,000 rows to keep things fast on a normal laptop
    df = pd.read_csv('paysim.csv', nrows=200000)
    print("✅ Dataset loaded from file.")
except FileNotFoundError:
    # If the file is not found, we generate synthetic data for demonstration
    print("⚠️  paysim.csv not found. Generating synthetic demo data...")
    np.random.seed(42)
    n = 200000
    fraud_ratio = 0.013  # ~1.3% fraud rate, similar to PaySim

    transaction_types = np.random.choice(
        ['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN'],
        size=n,
        p=[0.35, 0.25, 0.25, 0.10, 0.05]
    )

    # Amounts: fraudulent transactions tend to have larger amounts
    is_fraud_flag = np.random.choice([0, 1], size=n, p=[1-fraud_ratio, fraud_ratio])
    amounts = np.where(
        is_fraud_flag == 1,
        np.random.lognormal(mean=10, sigma=1.5, size=n),   # Fraudulent: larger
        np.random.lognormal(mean=7, sigma=1.2, size=n)     # Normal: smaller
    )

    df = pd.DataFrame({
        'step': np.random.randint(1, 744, size=n),          # Hour of simulation (1-744)
        'type': transaction_types,
        'amount': amounts.round(2),
        'nameOrig': [f'C{np.random.randint(1e8,1e9)}' for _ in range(n)],
        'oldbalanceOrg': np.random.uniform(0, 50000, size=n).round(2),
        'newbalanceOrig': np.random.uniform(0, 50000, size=n).round(2),
        'nameDest': [f'C{np.random.randint(1e8,1e9)}' for _ in range(n)],
        'oldbalanceDest': np.random.uniform(0, 100000, size=n).round(2),
        'newbalanceDest': np.random.uniform(0, 100000, size=n).round(2),
        'isFraud': is_fraud_flag,
        'isFlaggedFraud': np.zeros(n, dtype=int)
    })
    print(f"✅ Synthetic demo dataset created with {n:,} rows.")

print(f"\n📊 Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nFraud rate: {df['isFraud'].mean()*100:.2f}%")
df.head()

In [ ]:
# --- Understand the columns ---

column_guide = {
    'step':             'Hour of the simulation (1 = hour 1, 744 = last hour of month)',
    'type':             'Type of transaction: PAYMENT, TRANSFER, CASH_OUT, DEBIT, CASH_IN',
    'amount':           'Amount of money moved in this transaction',
    'nameOrig':         'ID of the account that sent money',
    'oldbalanceOrg':    'Sender balance BEFORE this transaction',
    'newbalanceOrig':   'Sender balance AFTER this transaction',
    'nameDest':         'ID of the account that received money',
    'oldbalanceDest':   'Receiver balance BEFORE this transaction',
    'newbalanceDest':   'Receiver balance AFTER this transaction',
    'isFraud':          '>>> TARGET: 1 = this transaction is money laundering, 0 = clean',
    'isFlaggedFraud':   'Simple rule-based flag from the original system (mostly unused)'
}

print("📋 Column Guide:")
print("-" * 70)
for col, desc in column_guide.items():
    print(f"  {col:<20} → {desc}")

print("\n📈 Basic info:")
df.info()

In [ ]:
# --- Look at the class imbalance visually ---

fraud_counts = df['isFraud'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Clean (0)', 'Fraud (1)'], fraud_counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Transaction Count by Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    fraud_counts.values,
    labels=['Clean', 'Fraud'],
    autopct='%1.2f%%',
    colors=['steelblue', 'tomato'],
    startangle=90
)
axes[1].set_title('Class Distribution')

plt.suptitle('⚠️  Severe Class Imbalance — This is why Accuracy is misleading!', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nClean transactions: {fraud_counts[0]:,} ({fraud_counts[0]/len(df)*100:.1f}%)")
print(f"Fraud transactions: {fraud_counts[1]:,} ({fraud_counts[1]/len(df)*100:.1f}%)")

---
## 4. Data Preprocessing

Before we train any model, we need to **clean and prepare** the data. Raw data is almost never ready for machine learning directly.

### 4.1 Missing Values

#### 🟢 Simple Explanation
Some rows might have empty cells — like a form where someone forgot to fill in a field. ML models usually can't handle empty values, so we need to deal with them first.

#### 🔵 Expert Insight: Impact on Model Bias
How you handle missing data is a critical architectural decision. Simply **dropping rows** with missing values seems safe, but in AML data, missingness is often *not random* — a missing balance field may correlate with fraudulent behavior (e.g., a mule account that was just opened). Blindly dropping those rows introduces **selection bias** and can cause the model to systematically miss a fraud pattern.

For PaySim, missing values are rare or non-existent (it's synthetic). But we document the process because in real production data, this step is often the most labor-intensive.

In [ ]:
# --- Step 1: Check for missing values ---

missing_counts = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_percent.round(3)
})

print("🔍 Missing Value Report:")
print(missing_summary)

# Drop rows with any missing values (safe for PaySim since there are none,
# but in real data you'd think more carefully about imputation strategy)
rows_before = len(df)
df = df.dropna()
rows_after = len(df)

print(f"\n✅ Rows before: {rows_before:,} | Rows after: {rows_after:,} | Dropped: {rows_before - rows_after:,}")

### 4.2 Feature Engineering & Dropping Unnecessary Columns

#### 🟢 Simple Explanation
We remove columns that don't help us learn anything (like account IDs — every ID is unique, so the model can't learn from them). We also create new features that might be useful.

#### 🔵 Expert Insight
Account IDs (`nameOrig`, `nameDest`) are high-cardinality identifiers. Including them raw causes **data leakage** in a time-split evaluation and also causes the model to memorize account numbers rather than learning behavioral patterns. In production, these IDs are used to *join* with account-level feature stores (aggregated historical behavior), not fed raw into the model.

In [ ]:
# --- Engineer new useful features ---

# Feature 1: How much did the sender's balance change?
# A legitimate transaction should reduce the sender's balance by exactly the amount sent.
# If it doesn't match, something suspicious might be going on.
df['balance_change_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig']

# Feature 2: How much did the receiver's balance change?
df['balance_change_dest'] = df['newbalanceDest'] - df['oldbalanceDest']

# Feature 3: The "error" between amount sent and balance change
# In a real transaction, amount ≈ balance_change_orig.
# A big discrepancy is a red flag (money may have been split/shuffled).
df['amount_balance_error'] = abs(df['amount'] - df['balance_change_orig'])

# Feature 4: Did the destination account start with zero balance? (mule account sign)
df['dest_was_empty'] = (df['oldbalanceDest'] == 0).astype(int)

# --- Drop columns that won't help the model ---
columns_to_drop = [
    'nameOrig',       # Account ID — too unique to learn from
    'nameDest',       # Account ID — too unique to learn from
    'isFlaggedFraud'  # The old rule-based flag — not useful as a training feature
]
df = df.drop(columns=columns_to_drop)

print(f"✅ Feature engineering complete.")
print(f"\nNew features added: balance_change_orig, balance_change_dest, amount_balance_error, dest_was_empty")
print(f"Dropped columns: {columns_to_drop}")
print(f"\nDataFrame shape: {df.shape}")
df.head(3)

### 4.3 Encoding Categorical Variables

#### 🟢 Simple Explanation
Machine learning models only understand numbers. The `type` column has text values like 'PAYMENT' or 'TRANSFER'. We need to convert these into numbers so the model can process them.

#### 🔵 Expert Insight
We use **Label Encoding** here because `type` has a small number of distinct values (5) and tree-based models (XGBoost) handle label-encoded categoricals well — they split on thresholds and don't assume any ordinal relationship between the numbers. If we were using a linear model or distance-based model, we'd use **One-Hot Encoding** instead to avoid implying a false ordering (e.g., that PAYMENT=0 is "less than" TRANSFER=1).

In [ ]:
# --- Encode the 'type' column (text → number) ---

# Show what values exist before encoding
print("Transaction types before encoding:")
print(df['type'].value_counts())

# Create the encoder
label_encoder = LabelEncoder()

# Fit and transform: learns the mapping, then applies it
df['type_encoded'] = label_encoder.fit_transform(df['type'])

# Show the mapping so we understand what happened
print("\nEncoding mapping:")
for label, number in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"  {label} → {number}")

# Drop the original text column (we no longer need it)
df = df.drop(columns=['type'])

print("\n✅ Encoding complete!")

### 4.4 Feature Scaling

#### 🟢 Simple Explanation
Imagine comparing a person's age (20–80) with their salary (20,000–200,000). The salary numbers are thousands of times bigger, so without scaling, a model might think salary is way more important just because the numbers are larger. Scaling puts all features on the same playing field.

#### 🔵 Expert Insight: Effect on Distance-Based Models
Scaling is **mandatory** for PCA and K-Means — both are distance-based. PCA finds directions of maximum variance; if `amount` has variance of 10⁸ and `step` has variance of 10², PCA will almost entirely be about `amount`. K-Means clusters by Euclidean distance, so an unscaled high-variance feature will completely dominate the cluster assignments.

XGBoost is tree-based and technically **doesn't require scaling** (trees split on thresholds, not distances). But we scale here anyway because we'll use the scaled version as input to PCA → K-Means → XGBoost to maintain a clean, consistent pipeline.

In [ ]:
# --- Separate features (X) and target (y) ---

# y = what we want to predict (is this fraud or not?)
y = df['isFraud']

# X = everything else (our inputs to the model)
X = df.drop(columns=['isFraud'])

print(f"Features (X): {X.shape}")
print(f"Target (y):   {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

# --- Apply Standard Scaling ---
# StandardScaler transforms each feature to have mean=0 and std=1
scaler = StandardScaler()

# Fit = learn the mean and std of each column
# Transform = apply the scaling formula: (value - mean) / std
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame so it's easier to read
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("\n✅ Scaling complete!")
print("\nBefore scaling (original):")
print(X[['amount', 'step', 'oldbalanceOrg']].describe().round(2))
print("\nAfter scaling (should all have similar range):")
print(X_scaled_df[['amount', 'step', 'oldbalanceOrg']].describe().round(2))

---
## 5. PCA — Dimensionality Reduction

### 🟢 Simple Explanation
Imagine you have a 3D object — a ball. You can describe it with x, y, z coordinates. But to make a simpler model, you could project (flatten) it onto a 2D surface and capture most of the information.

PCA does this mathematically. It finds the most "informative" directions in your data and projects everything onto fewer dimensions, keeping the most important variation.

### 🔵 Expert Insight: Variance Preservation
PCA doesn't just reduce dimensions arbitrarily — it finds the **principal components**, which are orthogonal (uncorrelated) directions that capture decreasing amounts of variance.

In AML, many features are highly correlated: `oldbalanceOrg`, `newbalanceOrig`, and `balance_change_orig` all describe the same underlying thing — the sender's money. PCA identifies this redundancy and collapses correlated features into independent components. This makes K-Means clustering much more effective because the algorithm is no longer confused by correlated, redundant features.

The **scree plot** and **cumulative explained variance** help us choose how many components to keep. A common rule: keep enough components to explain **≥85-95% of variance**.

In [ ]:
# --- Apply PCA ---

# First, run PCA keeping ALL components so we can see how much variance each explains
pca_full = PCA()
pca_full.fit(X_scaled)  # Learn from the scaled data

# How much variance does each component explain?
explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# --- Plot the explained variance ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Scree Plot (how much each component adds)
axes[0].bar(range(1, len(explained_variance) + 1), explained_variance * 100, color='steelblue')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Scree Plot — Individual Component Variance')
axes[0].set_xticks(range(1, len(explained_variance) + 1))

# Plot 2: Cumulative Explained Variance
axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance * 100,
             marker='o', color='tomato', linewidth=2)
axes[1].axhline(y=85, color='green', linestyle='--', label='85% threshold')
axes[1].axhline(y=95, color='orange', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Principal Components')
axes[1].set_ylabel('Cumulative Variance Explained (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].set_xticks(range(1, len(cumulative_variance) + 1))

plt.suptitle('PCA — How Much Information Does Each Component Carry?', fontsize=13)
plt.tight_layout()
plt.show()

# Print exact values
print("Cumulative variance explained by n components:")
for i, cv in enumerate(cumulative_variance, 1):
    marker = " ← choose this" if cv >= 0.85 and cumulative_variance[i-2] < 0.85 else ""
    print(f"  {i} components → {cv*100:.1f}% variance{marker}")

In [ ]:
# --- Choose the number of components and apply final PCA ---

# We want to keep at least 85% of the variance.
# Based on the plot above, find the first component count that crosses 85%.
n_components = int(np.argmax(cumulative_variance >= 0.85)) + 1

print(f"🎯 Chosen number of PCA components: {n_components}")
print(f"   → These {n_components} components explain {cumulative_variance[n_components-1]*100:.1f}% of the data's variance.")
print(f"   → We are discarding {(1 - cumulative_variance[n_components-1])*100:.1f}% of variance (noise/redundancy).")

# Apply PCA with the chosen number of components
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)  # Transform the data

# Name the new PCA columns
pca_col_names = [f'PC{i+1}' for i in range(n_components)]
X_pca_df = pd.DataFrame(X_pca, columns=pca_col_names)

print(f"\n✅ PCA applied! Shape went from {X_scaled.shape} → {X_pca.shape}")
X_pca_df.head()

---
## 6. K-Means Clustering — Behavioral Segmentation

### 🟢 Simple Explanation
K-Means is like sorting people into groups by similarity. Imagine you have 1,000 students and you want to group them into 5 study groups based on their scores. K-Means automatically finds those groups by looking at who is most similar to whom.

Here, we're grouping **transactions** by behavior — without being told which ones are fraud. This is called **unsupervised learning**.

### 🔵 Expert Insight: Behavioral Profiling in AML
Clustering serves a deeper strategic purpose in AML beyond just grouping. By segmenting accounts into behavioral clusters, we create a **behavioral baseline**. Any transaction that doesn't fit its account's typical cluster profile becomes a signal of anomaly.

In production, the cluster assignment (`behavioral_segment`) becomes an **engineered feature** passed to the supervised classifier. This is powerful because it provides the XGBoost model with context: "this transaction is from an account that typically behaves like Cluster 2, but this specific transaction looks like Cluster 4 behavior" — a cross-cluster pattern is a strong fraud signal.

In [ ]:
# --- Elbow Method to Find the Best K ---
# We try different values of K and measure the "inertia"
# Inertia = sum of distances from each point to its cluster center
# Lower inertia = tighter, more compact clusters

inertia_values = []  # Store inertia for each K
k_range = range(2, 11)  # Try K from 2 to 10

print("Running K-Means for different values of K... (this may take a moment)")

for k in k_range:
    # Train a K-Means model with k clusters
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=5)  # n_init=5 for speed
    kmeans_test.fit(X_pca)  # Use PCA-reduced data
    inertia_values.append(kmeans_test.inertia_)
    print(f"  K={k}: inertia={kmeans_test.inertia_:,.0f}")

# --- Plot the Elbow Curve ---
plt.figure(figsize=(9, 5))
plt.plot(list(k_range), inertia_values, marker='o', linewidth=2, color='steelblue')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Lower = Better Fit)')
plt.title('Elbow Method — Finding the Optimal Number of Clusters')
plt.xticks(list(k_range))
plt.annotate('The "elbow" is here\n→ diminishing returns after this point',
             xy=(4, inertia_values[2]), xytext=(6, inertia_values[2] * 1.05),
             arrowprops=dict(arrowstyle='->', color='tomato'), color='tomato')
plt.tight_layout()
plt.show()

In [ ]:
# --- Choose K and run final K-Means ---
# The elbow typically appears around K=4 or K=5 in PaySim.
# This makes intuitive sense: we'd expect roughly 4-5 behavioral types:
#   - Small regular payments
#   - Large business transfers
#   - Cash operations
#   - Unusual/outlier activity

CHOSEN_K = 4  # Adjust this based on what you see in the elbow plot above

print(f"🎯 Chosen K = {CHOSEN_K}")
print(f"   Reasoning: The elbow curve shows diminishing returns after K={CHOSEN_K}.")
print(f"   Adding more clusters improves inertia only marginally, increasing model complexity for little gain.")

# Train the final K-Means model
kmeans = KMeans(n_clusters=CHOSEN_K, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_pca)

# --- Add cluster labels back to our main dataset ---
# This is the new feature we'll pass to XGBoost!
X['behavioral_segment'] = cluster_labels
X_scaled_df['behavioral_segment'] = cluster_labels

print(f"\n✅ Clustering complete! Added 'behavioral_segment' column.")
print(f"\nCluster distribution:")
print(X['behavioral_segment'].value_counts().sort_index())

In [ ]:
# --- Visualize clusters using first 2 PCA components ---

plt.figure(figsize=(10, 6))

# Plot a sample of points (all 200k would be too slow to render)
sample_size = min(5000, len(X_pca))
sample_idx = np.random.choice(len(X_pca), size=sample_size, replace=False)

colors = ['steelblue', 'tomato', 'green', 'orange', 'purple']

for cluster_id in range(CHOSEN_K):
    # Find points that belong to this cluster
    mask = cluster_labels[sample_idx] == cluster_id
    plt.scatter(
        X_pca[sample_idx][mask, 0],  # PC1 (x-axis)
        X_pca[sample_idx][mask, 1],  # PC2 (y-axis)
        label=f'Cluster {cluster_id}',
        alpha=0.5,
        s=10,
        color=colors[cluster_id]
    )

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title(f'K-Means Clusters in PCA Space (K={CHOSEN_K}) — Sample of {sample_size:,} points')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Analyze fraud rate per cluster ---
# This tells us if some clusters are more suspicious than others

cluster_analysis = pd.DataFrame({
    'behavioral_segment': cluster_labels,
    'isFraud': y.values
})

fraud_by_cluster = cluster_analysis.groupby('behavioral_segment')['isFraud'].agg(['sum', 'count', 'mean'])
fraud_by_cluster.columns = ['Fraud Cases', 'Total Transactions', 'Fraud Rate']
fraud_by_cluster['Fraud Rate %'] = (fraud_by_cluster['Fraud Rate'] * 100).round(3)

print("📊 Fraud Rate by Behavioral Cluster:")
print(fraud_by_cluster)
print("\n💡 Clusters with higher fraud rates are more 'suspicious' behavioral groups.")
print("   XGBoost will learn to use this behavioral_segment feature as a signal!")

---
## 7. XGBoost Classification Model

### 🟢 Simple Explanation
Now we build the model that actually predicts whether a transaction is fraud. Think of it like hiring a detective who learns from thousands of past cases (training data) and then uses that knowledge to investigate new cases.

XGBoost works by building many small decision trees, where each tree learns from the mistakes of the previous one. Together they form a powerful team.

### 🔵 Expert Insight: Why XGBoost Dominates AML
XGBoost is the dominant algorithm for structured/tabular financial data for several reasons:

1. **Handles imbalanced data** natively via the `scale_pos_weight` parameter
2. **Resistant to outliers** (tree splits on ranks/thresholds, not raw values)
3. **Built-in regularization** (L1/L2) reduces overfitting
4. **Feature importance** is built in — critical for model explainability (regulators often require this)
5. **Partial/incremental learning** is supported — essential for our Continuous Learning section

In [ ]:
# --- Prepare Final Feature Matrix ---
# We use the ORIGINAL scaled features + the behavioral_segment cluster label

# Get the final feature set (all scaled features + behavioral_segment)
X_final = X_scaled_df.copy()
X_final['behavioral_segment'] = cluster_labels

print(f"Final feature matrix shape: {X_final.shape}")
print(f"Features used: {list(X_final.columns)}")

In [ ]:
# --- Split data into Training and Testing sets ---
# We use 80% of data to TRAIN the model,
# and 20% to TEST it on data it has NEVER seen before.

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.2,       # 20% for testing
    random_state=42,     # For reproducibility
    stratify=y           # Make sure fraud ratio is the same in both splits
)

print(f"Training set:  {X_train.shape[0]:,} rows")
print(f"Test set:      {X_test.shape[0]:,} rows")
print(f"\nFraud rate in training: {y_train.mean()*100:.2f}%")
print(f"Fraud rate in test:     {y_test.mean()*100:.2f}%")
print("✅ Stratification worked — both sets have the same fraud rate!")

In [ ]:
# --- Handle Class Imbalance ---
# Because fraud is rare (~1%), we tell XGBoost to pay more attention to fraud cases.
# scale_pos_weight = (number of clean transactions) / (number of fraud transactions)
# This makes the model treat each fraud case as if it were more important.

n_clean = (y_train == 0).sum()
n_fraud = (y_train == 1).sum()
scale_weight = n_clean / n_fraud

print(f"Clean transactions in training: {n_clean:,}")
print(f"Fraud transactions in training: {n_fraud:,}")
print(f"scale_pos_weight = {scale_weight:.1f}")
print(f"→ XGBoost will treat each fraud case as if it were {scale_weight:.0f} clean cases in importance")

In [ ]:
# --- Train the XGBoost Model ---
# Keeping parameters simple and student-friendly.
# No complex tuning — just sensible defaults.

# Convert data to XGBoost's special format (DMatrix) for efficiency
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(X_train.columns))
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=list(X_test.columns))

# Model parameters
params = {
    'objective':        'binary:logistic',  # Binary classification (fraud or not)
    'eval_metric':      'aucpr',            # Area under Precision-Recall curve (better than AUC for imbalanced)
    'max_depth':        6,                  # How deep each decision tree can grow
    'learning_rate':    0.1,                # How fast the model learns (lower = more careful)
    'subsample':        0.8,                # Use 80% of data for each tree (prevents overfitting)
    'colsample_bytree': 0.8,               # Use 80% of features for each tree
    'scale_pos_weight': scale_weight,       # Handle class imbalance
    'seed':             42                  # For reproducibility
}

# Train!
print("🚀 Training XGBoost model...")
evals_result = {}  # Store training progress

model = xgb.train(
    params,
    dtrain,
    num_boost_round=100,                    # Number of trees to build
    evals=[(dtrain, 'train'), (dtest, 'test')],
    evals_result=evals_result,
    verbose_eval=20                         # Print progress every 20 rounds
)

print("\n✅ Training complete!")

In [ ]:
# --- Plot Training Progress ---

train_scores = evals_result['train']['aucpr']
test_scores  = evals_result['test']['aucpr']

plt.figure(figsize=(10, 5))
plt.plot(train_scores, label='Training AUCPR', color='steelblue')
plt.plot(test_scores,  label='Validation AUCPR', color='tomato')
plt.xlabel('Training Round (Tree Number)')
plt.ylabel('AUC-PR Score (Higher = Better)')
plt.title('XGBoost Training Progress')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Final Training AUC-PR:   {train_scores[-1]:.4f}")
print(f"Final Validation AUC-PR: {test_scores[-1]:.4f}")

In [ ]:
# --- Feature Importance ---
# Which features did XGBoost find most useful?

importance_dict = model.get_score(importance_type='weight')  # How many times each feature was used
importance_df = pd.DataFrame(list(importance_dict.items()), columns=['Feature', 'Importance'])
importance_df = importance_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
plt.xlabel('Feature Importance (Number of times used in splits)')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

---
## 8. Model Evaluation

### 🟢 Simple Explanation
Now we check how well our model works. We test it on the 20% of data it has never seen before and measure its performance using three key metrics:

- **Precision**: Of all transactions the model flagged as fraud, how many were actually fraud? *(Don't waste analysts' time)*
- **Recall**: Of all actual fraud cases, how many did the model catch? *(Don't miss criminals)*
- **F1-score**: The balance between Precision and Recall.

### 🔵 Expert Insight: Metrics That Matter in AML

In AML, **Recall is the most critical metric** — the cost of missing a real money laundering case (False Negative) is orders of magnitude higher than investigating a false alarm:

- Missing a case → Regulatory fines ($100M+), criminal charges against bank officers, reputational destruction
- False alarm → An analyst spends 2 hours investigating a clean transaction

However, Recall cannot be maximized in isolation. A model that flags every single transaction as fraud has 100% Recall but 0% utility. This is why the **F1-score** and **Precision-Recall curve** are the right evaluation tools — not accuracy.

In [ ]:
# --- Make predictions on the test set ---

# The model outputs a probability (0.0 to 1.0) for each transaction
y_pred_prob = model.predict(dtest)

# We convert probabilities to 0 or 1 using a threshold
# Default threshold is 0.5, but in AML we might lower it to catch more fraud (increase recall)
THRESHOLD = 0.4  # Lower threshold = catch more fraud (higher recall, lower precision)
y_pred = (y_pred_prob >= THRESHOLD).astype(int)

print(f"Prediction threshold: {THRESHOLD}")
print(f"Transactions flagged as fraud: {y_pred.sum():,} ({y_pred.mean()*100:.2f}% of test set)")
print(f"Actual fraud in test set:      {y_test.sum():,} ({y_test.mean()*100:.2f}% of test set)")

In [ ]:
# --- Confusion Matrix ---
# Shows us exactly where the model is right and wrong

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt=',',
    cmap='Blues',
    xticklabels=['Predicted Clean', 'Predicted Fraud'],
    yticklabels=['Actually Clean', 'Actually Fraud']
)
plt.title('Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (TN): {tn:,}  → Correctly identified clean transactions")
print(f"False Positives (FP): {fp:,}  → Clean transactions wrongly flagged (wastes analyst time)")
print(f"False Negatives (FN): {fn:,}  → Missed fraud (DANGEROUS - this is what we minimize!)")
print(f"True Positives  (TP): {tp:,}  → Correctly caught fraud cases")

In [ ]:
# --- Full Classification Report ---

print("=" * 60)
print("         CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Clean (0)', 'Fraud (1)']))
print("=" * 60)

# Calculate individual metrics manually to understand them
if tp + fp > 0:
    precision = tp / (tp + fp)
else:
    precision = 0

if tp + fn > 0:
    recall = tp / (tp + fn)
else:
    recall = 0

if precision + recall > 0:
    f1 = 2 * (precision * recall) / (precision + recall)
else:
    f1 = 0

accuracy = (tp + tn) / (tp + tn + fp + fn)

print(f"\n📊 Manual Metric Calculation:")
print(f"  Accuracy  = {accuracy:.4f}  ← MISLEADING for imbalanced data!")
print(f"  Precision = {precision:.4f}  ← Of flagged transactions, {precision*100:.1f}% are real fraud")
print(f"  Recall    = {recall:.4f}  ← We caught {recall*100:.1f}% of all actual fraud cases")
print(f"  F1-Score  = {f1:.4f}  ← Balance between Precision and Recall")

print(f"\n💡 Why accuracy ({accuracy*100:.2f}%) is misleading:")
print(f"   A model that predicts EVERYTHING as clean would get {(1 - y_test.mean())*100:.2f}% accuracy")
print(f"   and catch ZERO fraudsters. Don't be fooled by high accuracy!")

---
## 9. Continuous Learning

### 🟢 Simple Explanation
Imagine a fraud detective who trained in 2020 and then stopped learning. By 2025, criminals have changed their tricks — and the detective is still looking for old patterns. They'll miss everything new!

A real ML model needs to keep learning from new data over time. This is called **Continuous Learning** or **Online Learning**.

Instead of retraining from scratch every month (which is slow and expensive), we **update the existing model** using only the new data.

### 🔵 Expert Insight: Concept Drift in Fraud Systems
**Concept drift** is the technical term for when the statistical relationship between features and the target changes over time. In AML, this happens constantly:

- Criminals adopt new payment channels (crypto, mobile money)
- They change transaction amounts/timing to stay below detection thresholds
- New typologies emerge (e.g., shell company networks evolve)

XGBoost's native `xgb.train()` supports **warm-starting** via the `xgb_model` parameter — it initializes from an existing model and adds new trees on top, trained on only the new data. This is the correct approach for batch incremental learning.

**Critical constraint**: We train on ONLY the new data — NOT the combined old + new data. This simulates a real production constraint where historical data may no longer be stored (privacy regulations like GDPR often require data deletion after a retention period).

In [ ]:
# --- Step 1: Simulate a NEW batch of incoming transaction data ---
# In production, this would be real new transactions from the following month.
# Here we simulate it by generating new synthetic data.

print("📦 Simulating new batch of transaction data (next month's data)...")

np.random.seed(99)  # Different seed = different data than training set
n_new = 20000       # 20,000 new transactions (smaller batch = realistic monthly update)

# Simulate slight concept drift: fraud patterns shift a little
# New fraud has slightly larger amounts on average (criminals adapting)
new_fraud_flag = np.random.choice([0, 1], size=n_new, p=[0.985, 0.015])  # Slightly higher fraud rate

new_amounts = np.where(
    new_fraud_flag == 1,
    np.random.lognormal(mean=10.5, sigma=1.5, size=n_new),  # Fraudsters now use larger amounts
    np.random.lognormal(mean=7, sigma=1.2, size=n_new)
)

new_types_raw = np.random.choice(
    ['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN'],
    size=n_new,
    p=[0.35, 0.25, 0.25, 0.10, 0.05]
)

new_old_bal_orig = np.random.uniform(0, 50000, n_new)
new_new_bal_orig = np.random.uniform(0, 50000, n_new)
new_old_bal_dest = np.random.uniform(0, 100000, n_new)
new_new_bal_dest = np.random.uniform(0, 100000, n_new)

# Build the new batch DataFrame
new_batch = pd.DataFrame({
    'step':               np.random.randint(745, 1488, n_new),  # Next month (hours 745-1488)
    'type':               new_types_raw,
    'amount':             new_amounts.round(2),
    'oldbalanceOrg':      new_old_bal_orig.round(2),
    'newbalanceOrig':     new_new_bal_orig.round(2),
    'oldbalanceDest':     new_old_bal_dest.round(2),
    'newbalanceDest':     new_new_bal_dest.round(2),
    'isFraud':            new_fraud_flag
})

print(f"✅ New batch created: {len(new_batch):,} rows")
print(f"   Fraud rate in new batch: {new_batch['isFraud'].mean()*100:.2f}%")
new_batch.head(3)

In [ ]:
# --- Step 2: Preprocess the new batch using the SAME transformers ---
# IMPORTANT: We use the SAME scaler, encoder, PCA, and KMeans fitted on the old data.
# We do NOT refit them — that would cause inconsistency between old and new features.

# Separate target
y_new = new_batch['isFraud']
X_new = new_batch.drop(columns=['isFraud'])

# Engineer same features as before
X_new['balance_change_orig']  = X_new['oldbalanceOrg'] - X_new['newbalanceOrig']
X_new['balance_change_dest']  = X_new['newbalanceDest'] - X_new['oldbalanceDest']
X_new['amount_balance_error'] = abs(X_new['amount'] - X_new['balance_change_orig'])
X_new['dest_was_empty']       = (X_new['oldbalanceDest'] == 0).astype(int)

# Encode 'type' using the SAME label encoder (transform only, not fit)
X_new['type_encoded'] = label_encoder.transform(X_new['type'])
X_new = X_new.drop(columns=['type'])

# Scale using the SAME scaler (transform only, not fit)
X_new_scaled = scaler.transform(X_new)
X_new_scaled_df = pd.DataFrame(X_new_scaled, columns=X_new.columns)

# Apply PCA using the SAME pca (transform only)
X_new_pca = pca.transform(X_new_scaled)

# Assign clusters using the SAME KMeans (predict only, not fit)
new_cluster_labels = kmeans.predict(X_new_pca)

# Add behavioral_segment to the new batch
X_new_scaled_df['behavioral_segment'] = new_cluster_labels

print(f"✅ New batch preprocessed using SAME transformers.")
print(f"   Shape: {X_new_scaled_df.shape}")

In [ ]:
# --- Step 3: Update the model using ONLY new data (no retraining from scratch!) ---

# Convert new batch to DMatrix format
d_new = xgb.DMatrix(
    X_new_scaled_df,
    label=y_new,
    feature_names=list(X_new_scaled_df.columns)
)

print("🔄 Updating existing model with new batch of data...")
print("   (This adds new trees to the existing model — NOT retraining from scratch!)")

# KEY STEP: xgb_model=model means we START from the existing model
# and add more trees on top using only the new data
updated_model = xgb.train(
    params,
    d_new,
    num_boost_round=20,         # Add 20 new trees (smaller than initial training)
    xgb_model=model,            # ← This is the key! Start from existing model weights
    verbose_eval=10
)

print("\n✅ Model updated successfully!")
print(f"   Initial model: 100 trees")
print(f"   Updated model: {updated_model.num_boosted_rounds()} trees total")

In [ ]:
# --- Step 4: Compare old model vs updated model on the new batch ---

# Predictions from the ORIGINAL model on new data
old_preds_prob = model.predict(d_new)
old_preds = (old_preds_prob >= THRESHOLD).astype(int)

# Predictions from the UPDATED model on new data
new_preds_prob = updated_model.predict(d_new)
new_preds = (new_preds_prob >= THRESHOLD).astype(int)

from sklearn.metrics import precision_score, recall_score, f1_score

print("📊 Performance Comparison on New Batch:")
print("-" * 55)
print(f"{'Metric':<15} {'Original Model':>18} {'Updated Model':>18}")
print("-" * 55)

for metric_name, metric_fn in [('Precision', precision_score), ('Recall', recall_score), ('F1-Score', f1_score)]:
    old_score = metric_fn(y_new, old_preds, zero_division=0)
    new_score = metric_fn(y_new, new_preds, zero_division=0)
    change = "↑" if new_score > old_score else ("↓" if new_score < old_score else "=")
    print(f"{metric_name:<15} {old_score:>18.4f} {new_score:>17.4f} {change}")

print("-" * 55)
print("\n💡 The updated model has adapted to the new fraud patterns in the latest batch.")
print("   This simulates what a monthly model refresh looks like in a real AML system.")

In [ ]:
# --- Optional: Save the updated model for future use ---

# Save the model (in production, this would go to a model registry like MLflow)
updated_model.save_model('aml_model_updated.json')

print("✅ Updated model saved as 'aml_model_updated.json'")
print("\n💡 In production: This file would be uploaded to a model registry (MLflow, SageMaker, etc.)")
print("   and automatically deployed to the scoring API with zero downtime.")

---
## 10. Conclusion

### 🟢 Simple Summary

Here's what we built, step by step:

1. **Loaded** financial transaction data (PaySim dataset)
2. **Cleaned** the data — handled missing values, removed irrelevant columns
3. **Engineered features** — created new signals like balance discrepancy
4. **Encoded and Scaled** data to prepare it for ML algorithms
5. **Applied PCA** to reduce dimensions while keeping 85%+ of information
6. **Clustered transactions** with K-Means to find behavioral groups
7. **Trained XGBoost** to classify fraud vs. clean transactions
8. **Evaluated** using Precision, Recall, and F1-score
9. **Updated the model** with a new batch of data — without retraining from scratch

---

### 🔵 Expert-Level Reflection on System Impact

What we've built is a **rudimentary version of a real AML detection system**. In production at a Tier-1 bank, this pipeline would involve significantly more complexity:

**What this pipeline gets right:**
- Layered approach (unsupervised → supervised) mirrors industry best practice
- Class imbalance is handled explicitly (not ignored)
- Continuous learning is implemented as partial training, not naive retraining
- Evaluation uses Recall + F1, not accuracy

**Real-world limitations to acknowledge:**

| Limitation | Production Solution |
|---|---|
| Single-model system | Ensemble of models + rule engine hybrid |
| No network analysis | Graph Neural Networks to catch structuring across accounts |
| No drift detection | Statistical drift monitors (e.g., PSI, KS-test) before every update |
| Manual threshold | Dynamic threshold tuned to analyst capacity constraints |
| No model explainability | SHAP values per prediction for regulatory audit trail |
| Batch updates | Streaming updates via Kafka + Flink for real-time detection |

**The business impact of getting this right:**
In 2023, global financial institutions paid over $6 billion in AML-related fines. A well-built ML system that reduces false positives by 30% can save a major bank tens of millions in analyst hours annually — while simultaneously improving the rate of criminal detection.

In [ ]:
# --- Final Pipeline Summary ---

print("""\n
╔══════════════════════════════════════════════════════════════════╗
║       AML DETECTION PIPELINE — COMPLETE SUMMARY                 ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  Step 1: Load Data           → PaySim / synthetic 200k rows     ║
║  Step 2: Handle Missing      → dropna() (none in PaySim)        ║
║  Step 3: Feature Engineering → 4 new financial signal features  ║
║  Step 4: Encode              → LabelEncoder for 'type' column   ║
║  Step 5: Scale               → StandardScaler (mean=0, std=1)   ║
║  Step 6: PCA                 → Reduce to n components (≥85% var)║
║  Step 7: K-Means             → K=4 clusters, behavioral_segment ║
║  Step 8: XGBoost             → Binary classifier, AUC-PR eval   ║
║  Step 9: Evaluate            → Precision, Recall, F1-score      ║
║  Step 10: Continuous Learn   → xgb_model= warm-start update     ║
║                                                                  ║
║  KEY INSIGHT: Recall > Precision in AML. Missing a fraudster    ║
║  costs 100× more than investigating a false alarm.              ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")